# Hybrid Descriptor Modeling

In this section, Morgan fingerprints are combined with physicochemical molecular descriptors to evaluate whether hybrid molecular representations improve permeability prediction performance.

# Hybrid Descriptor QSAR Modeling

This notebook evaluates hybrid molecular representations combining Morgan fingerprints and physicochemical descriptors for permeability prediction.

The workflow includes:
- Molecular fingerprint generation
- Physicochemical descriptor calculation
- Hybrid feature construction
- Machine learning benchmarking
- Model comparison

In [1]:
import pandas as pd

df = pd.read_csv("final_12k_log_transformed_papp_dataset.csv")

df.head()

,canonical_smiles,standard_value,log_papp
0,Br.Cc1c2c(cc[n+]1Cc1ccccc1)c1ccc(OCC(=O)OCCCCO...,0.05,-1.301030
1,Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1,3.49,0.542825
2,Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1,5.90,0.770852
3,Brc1ccc(C2(CC3CCCC3)c3ccccc3-c3nccn32)cn1,25.00,1.397940
4,Brc1ccc(C2(CC3CCOCC3)c3ccccc3-c3nccn32)cc1,31.00,1.491362


In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit import RDLogger

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RDLogger.DisableLog('rdApp.*')

## Morgan Fingerprint Generation

Morgan fingerprints are generated to capture molecular topology and structural subpatterns.

In [3]:
fingerprints = []

for smi in df['canonical_smiles']:

    mol = Chem.MolFromSmiles(smi)

    if mol:

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=2048
        )

        fingerprints.append(np.array(fp))

fingerprint_array = np.array(fingerprints)

print(fingerprint_array.shape)

(12290, 2048)


## Physicochemical Descriptor Generation

Key physicochemical descriptors related to permeability behavior are calculated using RDKit.

In [4]:
descriptor_list = []

for smi in df['canonical_smiles']:

    mol = Chem.MolFromSmiles(smi)

    if mol:

        descriptors = [
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol)
        ]

        descriptor_list.append(descriptors)

descriptor_array = np.array(descriptor_list)

print(descriptor_array.shape)

(12290, 6)


## Hybrid Feature Matrix Construction

Morgan fingerprints and physicochemical descriptors are combined to create a hybrid molecular representation for QSAR modeling.

In [5]:
X_hybrid = np.hstack([
    fingerprint_array,
    descriptor_array
])

y = df['log_papp'].values

print(X_hybrid.shape)

(12290, 2054)


## Train-Test Split

The hybrid molecular feature matrix is divided into training and testing sets to evaluate predictive performance on unseen molecules.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_hybrid,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(9832, 2054)
(2458, 2054)


## Random Forest Regression using Hybrid Features

A Random Forest model is trained using combined molecular fingerprints and physicochemical descriptors to evaluate whether hybrid molecular representations improve permeability prediction.

In [7]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
y_pred_rf = rf_model.predict(X_test)

In [9]:
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

r2_rf = r2_score(y_test, y_pred_rf)

print("Hybrid RF RMSE:", rmse_rf)
print("Hybrid RF R²:", r2_rf)

Hybrid RF RMSE: 0.5304087080829439
Hybrid RF R²: 0.6171705096469984


## XGBoost Regression using Hybrid Features

An XGBoost Regressor is trained using combined molecular fingerprints and physicochemical descriptors to evaluate nonlinear learning performance for permeability prediction.

In [10]:
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [11]:
y_pred_xgb = xgb_model.predict(X_test)

In [12]:
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

r2_xgb = r2_score(y_test, y_pred_xgb)

print("Hybrid XGBoost RMSE:", rmse_xgb)
print("Hybrid XGBoost R²:", r2_xgb)

Hybrid XGBoost RMSE: 0.5625899172681541
Hybrid XGBoost R²: 0.5693068316411349


## LightGBM Regression using Hybrid Features

A LightGBM Regressor is trained using combined molecular fingerprints and physicochemical descriptors to evaluate gradient boosting performance on hybrid molecular representations.

In [13]:
lgbm_model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.057012 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4885
[LightGBM] [Info] Number of data points in the train set: 9832, number of used features: 2029
[LightGBM] [Info] Start training from score 0.843078


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [14]:
y_pred_lgbm = lgbm_model.predict(X_test)

C:\Users\kiran\anaconda3\envs\chemtox_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [15]:
rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))

r2_lgbm = r2_score(y_test, y_pred_lgbm)

print("Hybrid LightGBM RMSE:", rmse_lgbm)
print("Hybrid LightGBM R²:", r2_lgbm)

Hybrid LightGBM RMSE: 0.5477957030571343
Hybrid LightGBM R²: 0.591660552623097


## Support Vector Regression using Hybrid Features

A Support Vector Regressor is trained using combined molecular fingerprints and physicochemical descriptors to evaluate nonlinear kernel-based learning for permeability prediction.

In [16]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [17]:
svr_model = SVR(
    kernel='rbf',
    C=10,
    gamma='scale'
)

svr_model.fit(X_train_scaled, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,10
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [18]:
y_pred_svr = svr_model.predict(X_test_scaled)

In [19]:
rmse_svr = np.sqrt(mean_squared_error(y_test, y_pred_svr))

r2_svr = r2_score(y_test, y_pred_svr)

print("Hybrid SVR RMSE:", rmse_svr)
print("Hybrid SVR R²:", r2_svr)

Hybrid SVR RMSE: 0.5451033347591289
Hybrid SVR R²: 0.5956645944533654


# Final Hybrid Descriptor QSAR Conclusions

This study evaluated the impact of combining Morgan fingerprints with physicochemical molecular descriptors for permeability prediction using multiple machine learning algorithms.

The objective was to investigate whether hybrid molecular representations improve QSAR model performance compared to fingerprint-only feature sets.

## Molecular Representation Strategy

Two molecular representation approaches were compared:

### 1. Fingerprint-Only Representation
- Morgan fingerprints (2048 bits)
- Structural topology-focused representation

### 2. Hybrid Representation
- Morgan fingerprints
- Physicochemical descriptors:
  - Molecular Weight (MolWt)
  - LogP
  - TPSA
  - Hydrogen Bond Donors
  - Hydrogen Bond Acceptors
  - Rotatable Bonds

The hybrid approach combined structural and physicochemical information into a unified molecular feature space.

## Models Benchmarked

The following machine learning models were evaluated:

- Random Forest Regressor
- XGBoost Regressor
- LightGBM Regressor
- Support Vector Regressor (SVR)

## Key Findings

### 1. Hybrid Descriptors Improved Predictive Performance
The addition of physicochemical descriptors improved predictive performance for most machine learning models.

This demonstrates that permeability prediction depends not only on molecular topology but also on global physicochemical properties such as polarity, lipophilicity, molecular size, and hydrogen bonding behavior.

### 2. Random Forest Achieved Best Overall Performance
The Hybrid Random Forest model achieved the best overall predictive performance:

- RMSE ≈ 0.53
- R² ≈ 0.62

This indicates that ensemble tree methods effectively integrated both structural fingerprints and physicochemical descriptors for permeability prediction.

### 3. Support Vector Regression Performance
SVR remained a strong performer and demonstrated excellent nonlinear learning capability. However, the addition of hybrid descriptors produced only modest improvements compared to the fingerprint-only SVR model.

### 4. Gradient Boosting Models
LightGBM showed moderate improvement after descriptor integration, while XGBoost demonstrated comparatively limited gains under the current parameter settings.

These results highlight that feature engineering benefits different algorithms differently depending on dataset characteristics and model architecture.

### 5. Importance of Molecular Representation
The study demonstrates that molecular representation plays a critical role in QSAR modeling performance.

Structural fingerprints capture local topology and substructure patterns, whereas physicochemical descriptors provide global molecular behavior relevant to permeability mechanisms.

Combining both representations produced a more informative feature space for machine learning.

## Overall Conclusion

The hybrid descriptor workflow successfully demonstrated that combining molecular fingerprints with physicochemical descriptors improves permeability prediction performance and enhances QSAR model robustness.

Among the evaluated models, the Hybrid Random Forest approach achieved the strongest predictive performance on the current permeability dataset.

The study further illustrates the importance of:
- feature engineering
- molecular representation design
- model benchmarking
- comparative QSAR analysis

for building effective cheminformatics machine learning pipelines.

This project establishes a strong foundation for future advanced studies involving:
- hyperparameter optimization
- cross-validation
- SHAP interpretation
- applicability domain analysis
- scaffold-aware validation
- graph neural networks
- deep learning-based molecular modeling

In [20]:
comparison_df = pd.DataFrame({

    'Model': [
        'Fingerprint RF',
        'Fingerprint XGBoost',
        'Fingerprint LightGBM',
        'Fingerprint SVR',
        'Hybrid RF',
        'Hybrid XGBoost',
        'Hybrid LightGBM',
        'Hybrid SVR'
    ],

    'RMSE': [
        0.550,
        0.570,
        0.555,
        0.546,
        0.530,
        0.563,
        0.548,
        0.545
    ],

    'R2': [
        0.588,
        0.557,
        0.581,
        0.595,
        0.617,
        0.569,
        0.592,
        0.596
    ]

})

comparison_df

,Model,RMSE,R2
0,Fingerprint RF,0.550,0.588
1,Fingerprint XGBoost,0.570,0.557
2,Fingerprint LightGBM,0.555,0.581
3,Fingerprint SVR,0.546,0.595
4,Hybrid RF,0.530,0.617
5,Hybrid XGBoost,0.563,0.569
6,Hybrid LightGBM,0.548,0.592
7,Hybrid SVR,0.545,0.596


In [21]:
comparison_df.sort_values(
    by='R2',
    ascending=False
)

,Model,RMSE,R2
4,Hybrid RF,0.530,0.617
7,Hybrid SVR,0.545,0.596
3,Fingerprint SVR,0.546,0.595
6,Hybrid LightGBM,0.548,0.592
0,Fingerprint RF,0.550,0.588
2,Fingerprint LightGBM,0.555,0.581
5,Hybrid XGBoost,0.563,0.569
1,Fingerprint XGBoost,0.570,0.557
